# Lab 4 — FastAPI Scoring API

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Wrap a trained KNN pipeline in a **FastAPI** application.
2. Define **Pydantic** request/response models for typed JSON I/O.
3. Expose `GET /health` and `POST /predict` endpoints.
4. Verify the API with **`TestClient`** (no live server required).

> **Checkpoints:** `GET /health` → **200** · `POST /predict` → **200** · `default_probability` ≈ **0.2**, `default_label` = **0**



## From notebook to production API

| Layer | Role |
|-------|------|
| **Trained model** | `Pipeline` with scaler + KNN — same artifact you'd pickle or log to MLflow |
| **Pydantic models** | Validate incoming JSON (types, ranges) before scoring |
| **FastAPI routes** | HTTP interface: health check + predict |
| **TestClient** | In-process HTTP tests — no `uvicorn` needed for the lab |

```text
Client JSON  →  LoanRequest  →  DataFrame  →  model.predict_proba  →  LoanPrediction JSON
```

---

## 1. Train the scoring model

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]
X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)
model.fit(X_train, y_train)
print(f"model trained on {len(X_train)} loans (k=5)")

---

## 2. Pydantic request and response models

In [ ]:
from pydantic import BaseModel, Field


class LoanRequest(BaseModel):
    loan_amnt: float = Field(..., gt=0)
    int_rate: float = Field(..., ge=0)
    annual_inc: float = Field(..., gt=0)
    dti: float = Field(..., ge=0)
    installment: float = Field(..., gt=0)


class LoanPrediction(BaseModel):
    default_probability: float
    default_label: int


sample_request = LoanRequest(
    loan_amnt=15000,
    int_rate=12.5,
    annual_inc=65000,
    dti=18.0,
    installment=450.0,
)
display(sample_request.model_dump())

`Field(..., gt=0)` rejects invalid inputs (e.g. negative loan amount) **before** the model runs — a key API safety layer.

---

## 3. Define the FastAPI app

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Lending Club Scoring API")


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


@app.post("/predict", response_model=LoanPrediction)
def predict(loan: LoanRequest) -> LoanPrediction:
    features = pd.DataFrame(
        [[loan.loan_amnt, loan.int_rate, loan.annual_inc, loan.dti, loan.installment]],
        columns=NUMERIC_FEATURES,
    )
    proba = float(model.predict_proba(features)[0][1])
    label = int(proba >= 0.5)
    return LoanPrediction(default_probability=round(proba, 4), default_label=label)

print("routes:", [route.path for route in app.routes])

---

## 4. Test with `TestClient` (no live server)

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

health = client.get("/health")
sample = {
    "loan_amnt": 15000,
    "int_rate": 12.5,
    "annual_inc": 65000,
    "dti": 18.0,
    "installment": 450.0,
}
response = client.post("/predict", json=sample)

print("Lab 4 — FastAPI scoring API")
print(f"GET /health -> {health.status_code} {health.json()}")
print(f"POST /predict -> {response.status_code}")
print(f"response body: {response.json()}")

---

## 5. Change the request — observe probability shift

In [ ]:
riskier = {
    "loan_amnt": 25000,
    "int_rate": 22.0,
    "annual_inc": 40000,
    "dti": 28.0,
    "installment": 850.0,
}
safer = sample.copy()

rows = []
for label, body in [("baseline", safer), ("riskier", riskier)]:
    r = client.post("/predict", json=body)
    rows.append({"profile": label, **r.json()})

display(pd.DataFrame(rows))

Higher `int_rate` and `dti` with lower income typically push `default_probability` up — the API returns a different score without retraining.

---

## 6. Validation error example (optional)

In [ ]:
bad = sample.copy()
bad["loan_amnt"] = -1000
bad_response = client.post("/predict", json=bad)
print(f"invalid loan_amnt -> HTTP {bad_response.status_code}")
print(bad_response.json())

Pydantic returns **422 Unprocessable Entity** for invalid input — the model never runs on bad data.

---

## 7. Live server (instructor demo)

To run outside the notebook:

```bash
cd hands-on/day-04/output
uvicorn lab04_fastapi_scoring_api:app --reload
```

Then POST to `http://127.0.0.1:8000/predict` with the same JSON body. Open `http://127.0.0.1:8000/docs` for the interactive Swagger UI.

---

## 8. Checkpoint summary

In [ ]:
body = response.json()
assert health.status_code == 200
assert health.json() == {"status": "ok"}
assert response.status_code == 200
assert body["default_label"] == 0
assert abs(body["default_probability"] - 0.2) < 0.05
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why separate `/health` from `/predict` in production deployments?
2. What would you add before exposing this API on the public internet?
3. How does Lab 6 (MLflow) relate to versioning the model behind this API?

**Previous:** [Lab 3 — Choose K](lab03_choose_k.ipynb)  
**Next:** [Lab 5 — FeatureTools auto FE](lab05_featuretools_auto_fe.ipynb)